# 13 — Expert review, the trial, and the model update

**What this is.** The loop that closes the system: a human looks at the proposals, a trial
gets cast, results come back, and the models are updated.

**Why it matters here.** This is the only place a mix can move above
`COMPUTATIONALLY_COMPLIANT`, and the only place the models are allowed to change.

**What breaks if it is ignored.** A human saying "we don't stock that admixture" gets
recorded as a failed experiment, and the model learns that a perfectly good mix does not
work.

---

## The eight outcomes, kept apart

The project instructions require these to stay distinct. Collapsing them is the single most
common way a self-driving laboratory poisons its own training set.

| outcome | trains the model? |
|---|---|
| `HUMAN_REJECTED_BEFORE_CAST` | **no** — nothing was measured |
| `MATERIAL_UNAVAILABLE` | **no** |
| `COST_REJECTED` | **no** |
| `CONSTRUCTABILITY_REJECTED` | **no** |
| `OTHER_PROJECT_CONSTRAINT` | **no** |
| `EXPERIMENT_FAILED` | **no** — it records that the cast failed, not a response value |
| `MEASURED_WORKABILITY` | **yes** — workability model |
| `MEASURED_STRENGTH` | **yes** — strength model |

**A human rejection without a physical experiment is not a numerical training
observation.** The reason is always recorded.

## The validation ladder

`[PC]` Set here and nowhere else. A model may never raise it.

```
COMPUTATIONALLY_COMPLIANT -> LAB_VALIDATED -> TRIAL_BATCH_VALIDATED -> PRODUCTION_VALIDATED
```

**Every mix this system has produced so far is level 1.** Nothing it has proposed has been
cast.

In [ ]:
# EXPORT: skip -- notebook preamble only
# Always load the CURRENT project modules from disk. Python imports a module once per
# kernel, so a kernel that was open before an earlier notebook rewrote layer0.py /
# screen.py / ... would silently keep the OLD code. Dropping them first makes every
# Run All pick up the latest files, with or without a kernel restart.
import sys as _sys
for _m in ("is_rules", "layer0", "workability_residual", "strength_residual",
           "screen", "selection", "loop"):
    _sys.modules.pop(_m, None)

import json
import math
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

import is_rules as IS
import layer0 as L0
import workability_residual as WR

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

In [ ]:
from enum import Enum


class Outcome(Enum):
    """What happened to a proposal. Trains the model only where stated."""

    HUMAN_REJECTED_BEFORE_CAST = "human_rejected_before_cast"
    MATERIAL_UNAVAILABLE = "material_unavailable"
    COST_REJECTED = "cost_rejected"
    CONSTRUCTABILITY_REJECTED = "constructability_rejected"
    OTHER_PROJECT_CONSTRAINT = "other_project_constraint"
    EXPERIMENT_FAILED = "experiment_failed"
    MEASURED_WORKABILITY = "measured_workability"
    MEASURED_STRENGTH = "measured_strength"


# [PC] The ONLY outcomes that may become numerical training observations.
TRAINING_OUTCOMES = frozenset({Outcome.MEASURED_WORKABILITY, Outcome.MEASURED_STRENGTH})


def trains_model(outcome) -> bool:
    """A human 'no' with no cast is not a data point. This is the enforcement."""
    return Outcome(outcome) in TRAINING_OUTCOMES


TRIAL_LOG_COLUMNS = [
    # identity
    "trial_id", "run_id", "plant_id", "state_hash", "proposed_at", "resolved_at",
    # the mix as designed
    "grade", "exposure", "use", "msa_mm", "cement_type", "slump_target_mm",
    "cement_kg_m3", "flyash_kg_m3", "ggbs_kg_m3", "water_kg_m3", "admixture_kg_m3",
    "binder_kg_m3", "fine_agg_kg_m3", "coarse_agg_kg_m3",
    "wb_ratio", "sp_pct_binder", "fine_frac_of_agg", "sand_zone",
    "target_mean", "S_used", "S_basis",
    # what the models said BEFORE the cast -- never overwritten
    "slump_pred", "slump_sd", "slump_lo90", "slump_hi90", "P_slump",
    "fc_pred", "fc_sd", "P_strength", "strength_status",
    # the code verdict
    "gate_pass", "gate_decided", "gate_hard_not_verified",
    "inside_training_range", "achievable_band",
    # what happened
    "outcome", "rejection_reason", "reviewer",
    "measured_slump_mm", "measured_fc_7d_mpa", "measured_fc_28d_mpa",
    "batch_id", "specimen_ids",
    # honesty
    "validation_level", "trains_model", "mode", "acquisition",
]


def new_trial_log() -> pd.DataFrame:
    return pd.DataFrame(columns=TRIAL_LOG_COLUMNS)


def log_proposals(batch, *, run_id, plant_id, req, state, mode, path="trial_log.csv"):
    """Write proposals to the log BEFORE any human sees them.

    Logging first is what makes a later rejection auditable: the prediction on record
    is the one the model made when it did not know the answer.
    """
    now = datetime.now(timezone.utc).isoformat(timespec="seconds")
    rows = []
    for i, r in batch.reset_index(drop=True).iterrows():
        row = {c: None for c in TRIAL_LOG_COLUMNS}
        row.update({
            "trial_id": f"{run_id}-T{i+1}", "run_id": run_id, "plant_id": plant_id,
            "state_hash": r.get("state_hash"), "proposed_at": now,
            "grade": req.grade, "exposure": req.exposure.name, "use": req.use.name,
            "msa_mm": state.msa_mm, "cement_type": state.cement_type,
            "slump_target_mm": req.target_slump_mm,
            "validation_level": L0.ValidationLevel.COMPUTATIONALLY_COMPLIANT.name,
            "trains_model": False, "mode": mode,
            "acquisition": r.get("acquisition"),
        })
        for c in TRIAL_LOG_COLUMNS:
            if c in r and row.get(c) is None:
                row[c] = r[c]
        rows.append(row)

    df = pd.DataFrame(rows, columns=TRIAL_LOG_COLUMNS)
    p = Path(path)
    if p.exists():
        old = pd.read_csv(p)
        df = pd.concat([old, df], ignore_index=True)
    df.to_csv(p, index=False)
    return df


def record_outcome(log, trial_id, outcome, *, reviewer=None, reason=None,
                   measured_slump_mm=None, measured_fc_7d=None, measured_fc_28d=None,
                   batch_id=None, specimen_ids=None, path="trial_log.csv"):
    """Record what happened. Refuses to file a measurement without a value, and
    refuses to promote a validation level without a measurement behind it."""
    outcome = Outcome(outcome)
    i = log.index[log["trial_id"] == trial_id]
    if len(i) != 1:
        raise KeyError(f"trial_id {trial_id!r} matched {len(i)} rows")
    i = i[0]

    if outcome is Outcome.MEASURED_WORKABILITY and measured_slump_mm is None:
        raise ValueError("MEASURED_WORKABILITY needs a measured slump")
    if outcome is Outcome.MEASURED_STRENGTH and measured_fc_28d is None:
        raise ValueError("MEASURED_STRENGTH needs a measured 28-day strength")
    if outcome not in TRAINING_OUTCOMES and any(
            v is not None for v in (measured_slump_mm, measured_fc_7d, measured_fc_28d)):
        raise ValueError(
            f"{outcome.name} carries measurements. If something was cast and measured, "
            f"the outcome is a MEASURED_* one. Record the rejection separately."
        )
    if outcome is not Outcome.HUMAN_REJECTED_BEFORE_CAST and outcome not in TRAINING_OUTCOMES \
            and outcome is not Outcome.EXPERIMENT_FAILED and not reason:
        raise ValueError(f"{outcome.name} must record a reason")

    log.loc[i, "outcome"] = outcome.value
    log.loc[i, "reviewer"] = reviewer
    log.loc[i, "rejection_reason"] = reason
    log.loc[i, "resolved_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    log.loc[i, "measured_slump_mm"] = measured_slump_mm
    log.loc[i, "measured_fc_7d_mpa"] = measured_fc_7d
    log.loc[i, "measured_fc_28d_mpa"] = measured_fc_28d
    log.loc[i, "batch_id"] = batch_id
    log.loc[i, "specimen_ids"] = ";".join(map(str, specimen_ids)) if specimen_ids else None
    log.loc[i, "trains_model"] = trains_model(outcome)

    if outcome in TRAINING_OUTCOMES:
        log.loc[i, "validation_level"] = L0.ValidationLevel.LAB_VALIDATED.name
    # TRIAL_BATCH_VALIDATED and PRODUCTION_VALIDATED are promoted by promote(), which
    # requires plant evidence -- never by a measurement alone.

    log.to_csv(path, index=False)
    return log


def promote(log, trial_id, level, *, evidence, path="trial_log.csv"):
    """[PC] Raise a validation level. Requires written evidence, always."""
    level = L0.ValidationLevel(level) if not isinstance(level, L0.ValidationLevel) else level
    if not evidence:
        raise ValueError("a validation level may not be raised without recorded evidence")
    i = log.index[log["trial_id"] == trial_id][0]
    cur = L0.ValidationLevel[log.loc[i, "validation_level"]]
    if level.value != cur.value + 1:
        raise ValueError(f"levels are climbed one at a time: {cur.name} -> {level.name}")
    if cur is L0.ValidationLevel.COMPUTATIONALLY_COMPLIANT and not log.loc[i, "trains_model"]:
        raise ValueError("cannot reach LAB_VALIDATED without a measured result behind it")
    log.loc[i, "validation_level"] = level.name
    log.loc[i, "rejection_reason"] = f"{log.loc[i, 'rejection_reason'] or ''}|promoted: {evidence}".strip("|")
    log.to_csv(path, index=False)
    return log

## 2. Turning the log into training rows

`[PC]` Only rows with `trains_model = True`. `[PC]` Every specimen from one physical batch
carries the **same** group key, so leave-one-group-out keeps them together. Inflating the
sample by counting cubes is the leakage the project rules forbid.

In [ ]:
def training_rows(log, kind="workability"):
    """Extract trainable rows. Group key = the physical batch, not the specimen."""
    want = (Outcome.MEASURED_WORKABILITY if kind == "workability"
            else Outcome.MEASURED_STRENGTH)
    df = log[log["outcome"] == want.value].copy()
    if df.empty:
        return df
    # [PC] one physical batch = one experimental unit
    df["design_grp"] = df["batch_id"].fillna(df["trial_id"])
    if kind == "workability":
        df["slump_mm"] = pd.to_numeric(df["measured_slump_mm"], errors="coerce")
        df = df.dropna(subset=["slump_mm"])
    else:
        df["fc_28d"] = pd.to_numeric(df["measured_fc_28d_mpa"], errors="coerce")
        df = df.dropna(subset=["fc_28d"])
    return df


def independent_conditions(df, group_col="design_grp") -> int:
    """The number that goes in the report. Not the row count."""
    return int(df[group_col].nunique()) if len(df) else 0


def prediction_error_report(log):
    """What the model said before the cast, against what came back. Never overwritten."""
    d = log[log["outcome"] == Outcome.MEASURED_WORKABILITY.value].copy()
    if d.empty:
        return pd.DataFrame()
    d["measured_slump_mm"] = pd.to_numeric(d["measured_slump_mm"], errors="coerce")
    d["slump_pred"] = pd.to_numeric(d["slump_pred"], errors="coerce")
    d = d.dropna(subset=["measured_slump_mm", "slump_pred"])
    d["error_mm"] = d["slump_pred"] - d["measured_slump_mm"]
    d["inside_90"] = ((d["measured_slump_mm"] >= pd.to_numeric(d["slump_lo90"], errors="coerce")) &
                      (d["measured_slump_mm"] <= pd.to_numeric(d["slump_hi90"], errors="coerce")))
    return d[["trial_id", "slump_pred", "slump_lo90", "slump_hi90",
              "measured_slump_mm", "error_mm", "inside_90"]]


def plant_tau(log, plant_id):
    """[PC] Plant offset tau: the systematic slump bias at one plant, in mm.

    Estimated only from that plant's MEASURED rows. It is an offset, not a rescaling,
    and it is applied to the prediction, never folded into the training target -- so a
    later refit is not contaminated by it.
    """
    d = log[(log["plant_id"] == plant_id) &
            (log["outcome"] == Outcome.MEASURED_WORKABILITY.value)].copy()
    d["measured_slump_mm"] = pd.to_numeric(d["measured_slump_mm"], errors="coerce")
    d["slump_pred"] = pd.to_numeric(d["slump_pred"], errors="coerce")
    d = d.dropna(subset=["measured_slump_mm", "slump_pred"])
    n_groups = independent_conditions(d.assign(design_grp=d["batch_id"].fillna(d["trial_id"])))
    if n_groups < 3:
        return {"plant_id": plant_id, "tau_mm": None, "n_groups": n_groups,
                "status": "NOT_VERIFIED -- fewer than 3 independent batches at this plant"}
    resid = (d["measured_slump_mm"] - d["slump_pred"]).to_numpy(float)
    return {"plant_id": plant_id, "tau_mm": float(np.mean(resid)),
            "sd_mm": float(np.std(resid, ddof=1)), "n_groups": n_groups,
            "status": "estimated"}


def refit_needed(log, wmodel, *, min_new_groups=2):
    """[PC] When is a refit worth running? When enough NEW independent batches exist."""
    tr = training_rows(log, "workability")
    have = set(map(str, wmodel.get("trained_on_groups", [])))
    new = set(map(str, tr["design_grp"].unique())) - have if len(tr) else set()
    return {"new_groups": sorted(new), "n_new": len(new),
            "refit": len(new) >= min_new_groups,
            "rule": f"[PC] refit once {min_new_groups} new independent batches exist"}

## 3. Walk the loop once

A dry run on a throw-away log, so the rules can be seen enforcing themselves.

In [ ]:
# EXPORT: skip -- worked example
DEMO = "13_demo_trial_log.csv"
Path(DEMO).unlink(missing_ok=True)

batch = pd.read_csv("12_next_experiments.csv") if Path("12_next_experiments.csv").exists() else None
state = L0.MaterialState(cement_type="OPC", cement_grade=43, cement_sg=3.15,
                         flyash_sg=2.20, ggbs_sg=2.90,
                         sand_type="crushed", sand_sg=2.65, sand_passing_600um=45.0,
                         ca_sg=2.74, ca_shape="angular", msa_mm=20.0,
                         admixture_sg=1.08, admixture_dosage_range_pct=(0.6, 1.5))
req = L0.Requirement(grade="M30", target_slump_mm=120.0, exposure=IS.Exposure.SEVERE,
                     use=IS.UseClass.RCC, pumpable=True, pumpable_reduction_frac=1.0)

if batch is None or batch.empty:
    print("no batch from notebook 12; run it first")
else:
    log = log_proposals(batch, run_id="DEMO-0001", plant_id="plant_a", req=req,
                        state=state, mode="explore", path=DEMO)
    print(f"logged {len(log)} proposals, all at "
          f"{log['validation_level'].iloc[0]}, trains_model={log['trains_model'].iloc[0]}")

    # 1. a human says no -- this must NOT become a training row
    log = record_outcome(log, log["trial_id"].iloc[0], Outcome.MATERIAL_UNAVAILABLE,
                         reviewer="plant QC", reason="GGBS not stocked this month",
                         path=DEMO)
    print(f"\nT1 rejected (material): trains_model = {log['trains_model'].iloc[0]}")

    # 2. the guard rails
    for bad, why in (
        (dict(outcome=Outcome.MEASURED_WORKABILITY), "measurement with no value"),
        (dict(outcome=Outcome.COST_REJECTED, measured_slump_mm=120.0), "rejection carrying a measurement"),
        (dict(outcome=Outcome.COST_REJECTED), "rejection with no reason"),
    ):
        try:
            record_outcome(log.copy(), log["trial_id"].iloc[1], path=DEMO, **bad)
            print(f"  NOT REFUSED: {why}   <- this is a bug")
        except ValueError as e:
            print(f"  refused: {why}")

    # 3. a real cast
    log = record_outcome(log, log["trial_id"].iloc[1], Outcome.MEASURED_WORKABILITY,
                         reviewer="lab", measured_slump_mm=118.0,
                         batch_id="B-2026-0001", specimen_ids=["C1", "C2", "C3"],
                         path=DEMO)
    print(f"\nT2 cast and measured: trains_model = {log['trains_model'].iloc[1]}, "
          f"level = {log['validation_level'].iloc[1]}")

    # 4. the ladder cannot be skipped
    try:
        promote(log.copy(), log["trial_id"].iloc[1],
                L0.ValidationLevel.PRODUCTION_VALIDATED, evidence="looks fine", path=DEMO)
        print("  NOT REFUSED: level skip   <- this is a bug")
    except ValueError as e:
        print(f"  refused: {e}")

    print("\nprediction vs measurement (the record the model cannot rewrite):")
    pe = prediction_error_report(log)
    print(pe.to_string(index=False) if len(pe) else "  none yet")

    tr = training_rows(log, "workability")
    print(f"\ntrainable rows {len(tr)} across "
          f"{independent_conditions(tr)} independent batches")
    print(f"tau: {plant_tau(log, 'plant_a')}")
    print(f"refit: {refit_needed(log, joblib.load('workability_residual_model.joblib'))}")

## Export

In [ ]:
"""Emit loop.py from this notebook's own cells.

Same mechanism as 00_is_codes_layer.ipynb: the module is a concatenation of the
cells that were just executed, so what ships is exactly what was tested. Fixture
and export cells are excluded -- fixtures stay in the notebook where a human runs
them and reads the evidence.
"""
import json as _json
from pathlib import Path as _Path

NOTEBOOK = "13_trial_loop_and_update.ipynb"
MODULE   = "loop.py"

_Q = chr(34) * 3
_HEADER = "\n".join([
    _Q + "Expert review, the trial log, and the model update.",
    "",
    "GENERATED FILE - do not edit. Edit {nb} and re-run its export cell.",
    "",
    "Eight outcomes, kept apart. Only MEASURED_WORKABILITY and MEASURED_STRENGTH",
    "become training rows: a human rejection with no cast is not an observation.",
    "One physical batch is one experimental unit, however many cubes it produced.",
    "Validation levels are set here and nowhere else, and never by a model.",
    _Q,
    "from __future__ import annotations",
    "",
    "import math",
    "from datetime import datetime, timezone",
    "from enum import Enum",
    "from pathlib import Path",
    "",
    "import numpy as np",
    "import pandas as pd",
    "",
    "import is_rules as IS",
    "import layer0 as L0",
    "import workability_residual as WR",
    "", "",
])


def export(notebook: str = NOTEBOOK, path: str = MODULE) -> None:
    nb = _json.loads(_Path(notebook).read_text(encoding="utf-8"))
    out, skipped = [], 0
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        src = "".join(cell["source"])
        if ("def export(" in src or "# EXPORT: skip" in src
                or src.lstrip().startswith("def _t(") or src.lstrip().startswith("!")):
            skipped += 1
            continue
        lines = [ln for ln in src.split("\n")
                 if not ln.startswith(("from __future__", "import ", "from dataclasses",
                                       "from enum import", "from pathlib", "from typing",
                                       "import numpy", "import pandas"))
]
        body = "\n".join(lines).strip("\n")
        if body:
            out.append(body)

    _Path(path).write_text(_HEADER.format(nb=notebook) + "\n\n\n".join(out) + "\n",
                           encoding="utf-8")
    ns: dict = {}
    exec(compile(_Path(path).read_text(encoding="utf-8"), path, "exec"), ns)
    assert ns["trains_model"](ns["Outcome"].HUMAN_REJECTED_BEFORE_CAST) is False
    assert ns["trains_model"](ns["Outcome"].MEASURED_WORKABILITY) is True
    print(f"wrote {path}: {len(out)} cells merged, {skipped} skipped, import-checked OK")


export()

## Open items — carried, not resolved

1. `[PC]` `plant_tau` needs at least three independent batches at a plant before it
   reports a number. Below that it returns `NOT_VERIFIED`, which is the honest answer, not
   a missing feature.
2. `[PC]` `refit_needed` triggers on two new independent batches. That threshold is a
   project choice and should be revisited once the number of families is no longer five.
3. `[IS]` 7-day results steer the next pick; only the 28-day result is the verdict.
   `IS 456 cl 16` governs acceptance, and a 7-day number must never be entered as
   `measured_fc_28d_mpa`. The schema keeps them in separate columns for exactly this reason.
4. The existing `trial_log.csv` in this repository uses a wider schema from the previous
   optimiser. `TRIAL_LOG_COLUMNS` here is the schema the loop writes; migrating the old
   file is a separate task and it has not been done.